In [ ]:
import sympy as sp

t, x, y, z, eps = sp.symbols("t x y z eps", real=True)

coords = [t, x, y, z]
basis_names = ["dt", "dx", "dy", "dz"]

c = sp.symbols("c", positive=True, real=True)
kappa = sp.symbols("kappa", real=True)

a = sp.Function("a")(t)

Ax = sp.Function("A_x")(t, x, y, z)
Ay = sp.Function("A_y")(t, x, y, z)
Az = sp.Function("A_z")(t, x, y, z)

A = [Ax, Ay, Az]


class Form:
    def __init__(self, terms=None):
        self.terms = {}

        if terms:
            for indices, coefficient in terms.items():
                coefficient = sp.simplify(coefficient)

                if coefficient != 0:
                    self.terms[tuple(indices)] = coefficient

    def __add__(self, other):
        result = dict(self.terms)

        for indices, coefficient in other.terms.items():
            result[indices] = sp.simplify(
                result.get(indices, 0) + coefficient
            )

            if result[indices] == 0:
                result.pop(indices)

        return Form(result)

    __radd__ = __add__

    def __neg__(self):
        return Form({
            indices: -coefficient
            for indices, coefficient in self.terms.items()
        })

    def __sub__(self, other):
        return self + (-other)

    def __mul__(self, scalar):
        return Form({
            indices: sp.simplify(scalar * coefficient)
            for indices, coefficient in self.terms.items()
        })

    __rmul__ = __mul__

    def wedge(self, other):
        result = {}

        for I, coefficient_I in self.terms.items():
            for J, coefficient_J in other.terms.items():

                if set(I) & set(J):
                    continue

                combined = list(I) + list(J)

                inversions = sum(
                    1
                    for p in range(len(combined))
                    for q in range(p + 1, len(combined))
                    if combined[p] > combined[q]
                )

                sign = (-1) ** inversions
                indices = tuple(sorted(combined))

                result[indices] = sp.simplify(
                    result.get(indices, 0)
                    + sign * coefficient_I * coefficient_J
                )

        return Form(result)

    def d(self):
        result = Form()

        for indices, coefficient in self.terms.items():
            for mu, coordinate in enumerate(coords):

                derivative = sp.diff(coefficient, coordinate)

                if derivative != 0:
                    result += Form({
                        (mu,): derivative
                    }).wedge(
                        Form({
                            indices: 1
                        })
                    )

        return result

    def trunc(self, order=1):
        result = {}

        for indices, expression in self.terms.items():
            expression = sp.expand(expression)
            kept_expression = 0

            for term in sp.Add.make_args(expression):
                degree = term.as_powers_dict().get(eps, 0)

                if degree <= order:
                    kept_expression += term

            kept_expression = sp.simplify(kept_expression)

            if kept_expression != 0:
                result[indices] = kept_expression

        return Form(result)

    def __repr__(self):
        if not self.terms:
            return "0"

        parts = []

        for indices, coefficient in sorted(self.terms.items()):
            basis = "∧".join(
                basis_names[index]
                for index in indices
            )

            if indices == ():
                parts.append(f"({sp.sstr(coefficient)})")
            else:
                parts.append(
                    f"({sp.sstr(coefficient)}) {basis}"
                )

        return " + ".join(parts)


def oneform(index):
    return Form({
        (index,): 1
    })


dt = oneform(0)

dx = [
    oneform(1),
    oneform(2),
    oneform(3)
]


e0 = c * dt + sum(
    [
        eps * kappa * A[i] * dx[i]
        for i in range(3)
    ],
    Form()
)

ei = [
    a * dx[i]
    for i in range(3)
]


T0 = e0.d()

Ti = [
    ei[i].d()
    for i in range(3)
]


print("Original e^0:")
print(e0)

for i in range(3):
    print(f"Original e^{i + 1}:")
    print(ei[i])


print("\nOriginal T^0:")
print(T0)

for i in range(3):
    print(f"Original T^{i + 1}:")
    print(Ti[i])


beta = [
    -eps * kappa * A[i] / a
    for i in range(3)
]


ep0 = (
    e0
    + sum(
        [
            beta[i] * ei[i]
            for i in range(3)
        ],
        Form()
    )
).trunc(1)


epi = [
    (
        ei[i]
        + beta[i] * e0
    ).trunc(1)
    for i in range(3)
]


print("\nBoost parameters:")

for i in range(3):
    print(f"beta_{i + 1} =")
    sp.pprint(beta[i])


print("\nTransformed e'^0:")
print(ep0)

for i in range(3):
    print(f"Transformed e'^{i + 1}:")
    print(epi[i])


omega_i0 = [
    (
        -Form({
            (): beta[i]
        }).d()
    ).trunc(1)
    for i in range(3)
]

omega_0i = omega_i0


for i in range(3):
    print(f"\nomega'^{i + 1}_0:")
    print(omega_i0[i])


Tp0 = (
    ep0.d()
    + sum(
        [
            omega_0i[i].wedge(epi[i])
            for i in range(3)
        ],
        Form()
    )
).trunc(1)


Tpi = [
    (
        epi[i].d()
        + omega_i0[i].wedge(ep0)
    ).trunc(1)
    for i in range(3)
]


print("\nTransformed T'^0:")
print(Tp0)

for i in range(3):
    print(f"Transformed T'^{i + 1}:")
    print(Tpi[i])


Tcov0 = (
    T0
    + sum(
        [
            beta[i] * Ti[i]
            for i in range(3)
        ],
        Form()
    )
).trunc(1)


Tcovi = [
    (
        Ti[i]
        + beta[i] * T0
    ).trunc(1)
    for i in range(3)
]


print("\nCovariance check for T'^0:")

print("T'^0 =")
print(Tp0)

print("T^0 + beta_i T^i =")
print(Tcov0)

print("Difference =")
print((Tp0 - Tcov0).trunc(1))


print("\nCovariance check for T'^i:")

for i in range(3):
    print(f"\nT'^{i + 1} =")
    print(Tpi[i])

    print(f"T^{i + 1} + beta_{i + 1} T^0 =")
    print(Tcovi[i])

    print("Difference =")
    print((Tpi[i] - Tcovi[i]).trunc(1))

Original e^0:
(c) dt + (eps*kappa*A_x(t, x, y, z)) dx + (eps*kappa*A_y(t, x, y, z)) dy + (eps*kappa*A_z(t, x, y, z)) dz
Original e^1:
(a(t)) dx
Original e^2:
(a(t)) dy
Original e^3:
(a(t)) dz

Original T^0:
(eps*kappa*Derivative(A_x(t, x, y, z), t)) dt∧dx + (eps*kappa*Derivative(A_y(t, x, y, z), t)) dt∧dy + (eps*kappa*Derivative(A_z(t, x, y, z), t)) dt∧dz + (eps*kappa*(-Derivative(A_x(t, x, y, z), y) + Derivative(A_y(t, x, y, z), x))) dx∧dy + (eps*kappa*(-Derivative(A_x(t, x, y, z), z) + Derivative(A_z(t, x, y, z), x))) dx∧dz + (eps*kappa*(-Derivative(A_y(t, x, y, z), z) + Derivative(A_z(t, x, y, z), y))) dy∧dz
Original T^1:
(Derivative(a(t), t)) dt∧dx
Original T^2:
(Derivative(a(t), t)) dt∧dy
Original T^3:
(Derivative(a(t), t)) dt∧dz

Boost parameters:
beta_1 =
-eps⋅κ⋅Aₓ(t, x, y, z) 
──────────────────────
         a(t)         
beta_2 =
-eps⋅κ⋅A_y(t, x, y, z) 
───────────────────────
         a(t)          
beta_3 =
-eps⋅κ⋅A_z(t, x, y, z) 
───────────────────────
         a(t)       